In [45]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI


In [46]:
load_dotenv() 
 # Load environment variables from .env file
class llmState(TypedDict):
    question: str
    answer: str
    context: str


In [47]:
def llm_ask(state: llmState) -> llmState:

    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0.2
    )

    question = state["question"]
    context = state["context"]

    prompt = f"""
You are a conversational RAG assistant.

Previous context:
{context}

Current question:
{question}

Instructions:
- Answer the question using the previous context.
- Keep important information from the previous context.
- Do not forget important facts from previous turns.
- Update the context with any new important information.
- Remove only irrelevant or redundant information.
- Return ONLY valid JSON.

Return exactly:

{{
    "answer": "Your answer",
    "updated_context": "The complete important context to remember for future questions"
}}
"""

    import json

    response = llm.invoke(prompt)

    # Normalize content to a plain string
    content = response.content
    if isinstance(content, list):
        content = "".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        )

    # Strip markdown code fences if Gemini wraps the JSON in ```json ... ```
    content = content.strip()
    if content.startswith("```"):
        content = content.strip("`")
        content = content.split("\n", 1)[-1]  # drop language tag line
        if content.endswith("json"):
            content = content[:-4]

    data = json.loads(content)

    state["answer"] = data["answer"]
    state["context"] = data.get("updated_context", state["context"])

    return state

In [48]:
graph = StateGraph(llmState)

graph.add_node("llm_ask",llm_ask)

graph.add_edge(START,"llm_ask")
graph.add_edge("llm_ask",END)

workflow = graph.compile()



In [ ]:
context = ""

while True:
    user_input = input("Enter your question: ")

    if user_input.lower() == "exit":
        break

    state = {
        "question": user_input,
        "answer": "",
        "context": context
    }

    result = workflow.invoke(state)

    print("Answer:", result["answer"])
    print("Updated Context:", result["context"])

    context = result["context"]